In [1]:
import pymysql
from getpass import getpass

connection = pymysql.connect(
    host="127.0.0.1",
    port=3306,
    user=input("MySQL username: ").strip(),
    password=getpass("MySQL password: "),
)

print("Connected to MySQL!")

MySQL username:  root
MySQL password:  ········


Connected to MySQL!


In [2]:
with connection.cursor() as cursor:
    cursor.execute("CREATE DATABASE IF NOT EXISTS retail_inventory_project")
    cursor.execute("USE retail_inventory_project")
    cursor.execute("SELECT DATABASE()")
    print("Current database:", cursor.fetchone()[0])

Current database: retail_inventory_project


In [4]:
import csv
from pathlib import Path

# Find the cleaned CSV beside the notebook or in Downloads.
filename = "retail_store_inventory_cleaned.csv"
csv_path = Path.cwd() / filename

if not csv_path.is_file():
    csv_path = Path.home() / "Downloads" / filename

if not csv_path.is_file():
    raise FileNotFoundError(
        f"Put {filename} in Downloads or beside your notebook, then rerun this cell."
    )

with csv_path.open(newline="", encoding="utf-8-sig") as file:
    rows = list(csv.DictReader(file))

expected_count = 73_100
if len(rows) != expected_count:
    raise ValueError(f"Expected {expected_count:,} CSV rows, found {len(rows):,}.")

create_table_sql = """
CREATE TABLE IF NOT EXISTS retail_inventory_project.daily_inventory (
    `date` DATE NOT NULL,
    store_id CHAR(4) NOT NULL,
    product_id CHAR(5) NOT NULL,
    category VARCHAR(50) NOT NULL,
    region VARCHAR(30) NOT NULL,
    inventory_level INT NOT NULL,
    units_sold INT NOT NULL,
    units_ordered INT NOT NULL,
    demand_forecast DECIMAL(10, 2) NOT NULL,
    price DECIMAL(10, 2) NOT NULL,
    discount TINYINT UNSIGNED NOT NULL,
    weather_condition VARCHAR(30) NOT NULL,
    holiday_promotion TINYINT UNSIGNED NOT NULL,
    competitor_pricing DECIMAL(10, 2) NOT NULL,
    seasonality VARCHAR(20) NOT NULL,
    PRIMARY KEY (`date`, store_id, product_id)
) ENGINE=InnoDB
"""

insert_sql = """
INSERT INTO retail_inventory_project.daily_inventory (
    `date`, store_id, product_id, category, region,
    inventory_level, units_sold, units_ordered, demand_forecast,
    price, discount, weather_condition, holiday_promotion,
    competitor_pricing, seasonality
) VALUES (
    %(date)s, %(store_id)s, %(product_id)s, %(category)s, %(region)s,
    %(inventory_level)s, %(units_sold)s, %(units_ordered)s,
    %(demand_forecast)s, %(price)s, %(discount)s,
    %(weather_condition)s, %(holiday_promotion)s,
    %(competitor_pricing)s, %(seasonality)s
)
ON DUPLICATE KEY UPDATE store_id = daily_inventory.store_id
"""

try:
    with connection.cursor() as cursor:
        cursor.execute(create_table_sql)

        batch_size = 5_000
        for start in range(0, len(rows), batch_size):
            cursor.executemany(insert_sql, rows[start:start + batch_size])

        cursor.execute(
            "SELECT COUNT(*) FROM retail_inventory_project.daily_inventory"
        )
        row_count = cursor.fetchone()[0]

        if row_count != expected_count:
            raise ValueError(
                f"Expected {expected_count:,} database rows, found {row_count:,}."
            )

    connection.commit()

except Exception:
    connection.rollback()
    raise

print(f"Import verified: {row_count:,} rows in daily_inventory.")

Import verified: 73,100 rows in daily_inventory.


In [5]:
with connection.cursor() as cursor:
    cursor.execute("""
        SELECT
            COUNT(*) AS total_rows,
            COUNT(DISTINCT store_id) AS total_stores,
            COUNT(DISTINCT product_id) AS total_products,
            MIN(`date`) AS first_date,
            MAX(`date`) AS last_date
        FROM retail_inventory_project.daily_inventory
    """)
    total_rows, stores, products, first_date, last_date = cursor.fetchone()

print(f"Rows: {total_rows:,}")
print(f"Stores: {stores}")
print(f"Products: {products}")
print(f"Date range: {first_date} to {last_date}")

Rows: 73,100
Stores: 5
Products: 20
Date range: 2022-01-01 to 2024-01-01
